# Lab | Agent & Vector store

**Change the state union dataset and replicate this lab by updating the prompts accordingly.**

One such dataset is the [sonnets.txt](https://github.com/martin-gorner/tensorflow-rnn-shakespeare/blob/master/shakespeare/sonnets.txt) dataset or any other data of your choice from the same git.

# Combine agents and vector stores

This notebook covers how to combine agents and vector stores. The use case for this is that you've ingested your data into a vector store and want to interact with it in an agentic manner.

The recommended method for doing so is to create a `RetrievalQA` and then use that as a tool in the overall agent. Let's take a look at doing this below. You can do this with multiple different vector DBs, and use the agent as a way to route between them. There are two different ways of doing this - you can either let the agent use the vector stores as normal tools, or you can set `return_direct=True` to really just use the agent as a router.

## Create the vector store

In [1]:
!pip install chromadb langchain langchain_community langchain_openai langchain_text_splitters langchain-classic


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from langchain_classic.chains import RetrievalQA  # Legacy chains
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAI, OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.document_loaders import TextLoader

In [3]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [ ]:
# If you're using colab, run this
#os.environ['OPENAI_API_KEY'] = "YOUR_OPENAI_API_KEY"

In [4]:
llm = OpenAI(temperature=0)

In [5]:
from pathlib import Path

relevant_parts = []
for p in Path(".").absolute().parts:
    relevant_parts.append(p)
    if relevant_parts[-3:] == ["langchain", "docs", "modules"]:
        break
doc_path = str(Path(*relevant_parts) / "state_of_the_union.txt")


In [6]:
loader = TextLoader(doc_path, encoding="utf8")
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = text_splitter.split_documents(documents)

embeddings = OpenAIEmbeddings()
docsearch = Chroma.from_documents(texts, embeddings, collection_name="state-of-union")

In [7]:
state_of_union = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=docsearch.as_retriever()
)

In [8]:
from pathlib import Path

#  sonnets file is in the SAME folder as state_of_the_union.txt
sonnets_path = str(Path(doc_path).with_name("sonnets.txt")) 

sonnets_loader = TextLoader(sonnets_path, encoding="utf8")
sonnets_docs = sonnets_loader.load()

# Reuse the same text_splitter and embeddings you already created earlier
sonnets_texts = text_splitter.split_documents(sonnets_docs)

sonnets_db = Chroma.from_documents(sonnets_texts, embeddings, collection_name="sonnets")

sonnets = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=sonnets_db.as_retriever()
)

In [9]:
import os
os.environ["USER_AGENT"] = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"

In [10]:
from langchain_community.document_loaders import WebBaseLoader

In [11]:
loader = WebBaseLoader("https://beta.ruff.rs/docs/faq/")

In [12]:
%pip install beautifulsoup4 lxml html5lib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
# Put near where bs4 is imported
try:
    from bs4 import BeautifulSoup
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(
        "bs4 (beautifulsoup4) is required. Install with `pip install beautifulsoup4`"
    ) from e

In [14]:
docs = loader.load()
ruff_texts = text_splitter.split_documents(docs)
ruff_db = Chroma.from_documents(ruff_texts, embeddings, collection_name="ruff")
ruff = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=ruff_db.as_retriever()
)

Created a chunk of size 2122, which is longer than the specified 1000
Created a chunk of size 3187, which is longer than the specified 1000
Created a chunk of size 1017, which is longer than the specified 1000
Created a chunk of size 1256, which is longer than the specified 1000
Created a chunk of size 2321, which is longer than the specified 1000


## Create the Agent

In [15]:
# Import things that are needed generically
# Legacy agent imports
from langchain_classic.agents import AgentType, initialize_agent
from langchain_classic.tools import Tool  # or langchain.tools.Tool if using v1 Tool
from langchain_openai import OpenAI

In [16]:
tools = [
    Tool(
        name="State of Union QA System",
        func=state_of_union.run,
        description="useful for when you need to answer questions about the most recent state of the union address. Input should be a fully formed question.",
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question.",
    ),
    Tool(
        name="Sonnets QA System",
        func=sonnets.run,
        description="useful for answering questions about the sonnets text file (poetry). Ask about lines, themes, imagery, or which sonnet contains a quote. Input should be a fully formed question.",
    ),
]

In [17]:
# Construct the agent. We will use the default agent type here.
# See documentation for a full list of options.
agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

C:\Users\yogan\AppData\Local\Temp\ipykernel_31792\1834837320.py:3: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the [LangGraph documentation](https://langchain-ai.github.io/langgraph/) as well as guides for [Migrating from AgentExecutor](https://python.langchain.com/docs/how_to/migrate_agent/) and LangGraph's [Pre-built ReAct agent](https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/).
  agent = initialize_agent(


In [18]:
agent.invoke(
    "What did biden say about ketanji brown jackson in the state of the union address?"
)



> Entering new AgentExecutor chain...
 I should use the State of Union QA System to answer this question.
Action: State of Union QA System
Action Input: "What did biden say about ketanji brown jackson in the state of the union address?"
Observation:  Biden mentioned that he nominated Circuit Court of Appeals Judge Ketanji Brown Jackson to serve on the United States Supreme Court, praising her as one of the nation's top legal minds who will continue Justice Breyer's legacy of excellence.
Thought: I now know the final answer.
Final Answer: Biden nominated Ketanji Brown Jackson to serve on the Supreme Court and praised her as one of the nation's top legal minds.

> Finished chain.


{'input': 'What did biden say about ketanji brown jackson in the state of the union address?',
 'output': "Biden nominated Ketanji Brown Jackson to serve on the Supreme Court and praised her as one of the nation's top legal minds."}

In [19]:
agent.invoke("Why use ruff over flake8?")



> Entering new AgentExecutor chain...
 Ruff is a python linter that has some advantages over flake8, so it's worth considering.
Action: Ruff QA System
Action Input: Why use ruff over flake8?
Observation: 
There are a few reasons why someone might choose to use Ruff over Flake8:

1. Larger rule set: Ruff implements over 800 rules, while Flake8 only implements around 200. This means that Ruff can catch more potential issues in your code.

2. Better compatibility with other tools: Ruff is designed to work well with other tools like Black, isort, and type checkers like Mypy. This means that you can use Ruff alongside these tools to get more comprehensive feedback on your code.

3. Automatic fixing of lint violations: Unlike Flake8, Ruff is capable of automatically fixing its own lint violations. This can save you time and effort when fixing issues in your code.

4. Native implementation of popular Flake8 plugins: Ruff re-implements some of the most popular Flake8 plugins natively, which 

{'input': 'Why use ruff over flake8?',
 'output': 'Ruff offers a larger rule set, better compatibility with other tools, automatic fixing of lint violations, and native implementation of popular Flake8 plugins, making it a more comprehensive and user-friendly option compared to Flake8.'}

In [20]:
agent.invoke("Summarize the main theme of Sonnet 18.")



> Entering new AgentExecutor chain...
 The Sonnets QA System would be the best tool to use for this question since it is specifically designed for answering questions about poetry.
Action: Sonnets QA System
Action Input: "What is the main theme of Sonnet 18?"
Observation:  The main theme of Sonnet 18 is the speaker's comparison of the subject to a summer's day and the idea that the subject's beauty will never fade.
Thought: This answer seems accurate based on my knowledge of Sonnet 18.
Final Answer: The main theme of Sonnet 18 is the speaker's comparison of the subject to a summer's day and the idea that the subject's beauty will never fade.

> Finished chain.


{'input': 'Summarize the main theme of Sonnet 18.',
 'output': "The main theme of Sonnet 18 is the speaker's comparison of the subject to a summer's day and the idea that the subject's beauty will never fade."}

In [21]:
agent.invoke("Which sonnet talks about 'the marriage of true minds'?")



> Entering new AgentExecutor chain...
 I should use the Sonnets QA System to find the answer.
Action: Sonnets QA System
Action Input: Which sonnet talks about 'the marriage of true minds'?
Observation:  Sonnet CXVII talks about 'the marriage of true minds'.
Thought: I now know the final answer.
Final Answer: Sonnet CXVII

> Finished chain.


{'input': "Which sonnet talks about 'the marriage of true minds'?",
 'output': 'Sonnet CXVII'}

## Use the Agent solely as a router

You can also set `return_direct=True` if you intend to use the agent as a router and just want to directly return the result of the RetrievalQAChain.

Notice that in the above examples the agent did some extra work after querying the RetrievalQAChain. You can avoid that and just return the result directly.

In [22]:
tools = [
    Tool(
        name="State of Union QA System",
        func=state_of_union.run,
        description="useful for when you need to answer questions about the most recent state of the union address. Input should be a fully formed question.",
        return_direct=True,
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question.",
        return_direct=True,
    ),
    Tool(
        name="Sonnets QA System",
        func=sonnets.run,
        description="useful for answering questions about the sonnets text file (poetry). Ask about lines, themes, imagery, or which sonnet contains a quote. Input should be a fully formed question.",
    ),
]

In [23]:
agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

In [24]:
agent.invoke(
    "What did biden say about ketanji brown jackson in the state of the union address?"
)



> Entering new AgentExecutor chain...
 I should use the State of Union QA System to answer this question.
Action: State of Union QA System
Action Input: "What did Biden say about Ketanji Brown Jackson in the state of the union address?"
Observation:  Biden said that he nominated Ketanji Brown Jackson for the United States Supreme Court and praised her as one of the nation's top legal minds who will continue Justice Breyer's legacy of excellence.


> Finished chain.


{'input': 'What did biden say about ketanji brown jackson in the state of the union address?',
 'output': " Biden said that he nominated Ketanji Brown Jackson for the United States Supreme Court and praised her as one of the nation's top legal minds who will continue Justice Breyer's legacy of excellence."}

In [26]:
agent.invoke("Why use ruff over flake8?")



> Entering new AgentExecutor chain...
 It's important to understand the differences between different linters and their capabilities.
Action: Ruff QA System
Action Input: "What are the advantages of using ruff over flake8?"
Observation:  Ruff implements more rules than Flake8, has better compatibility with Black, and can automatically fix its own lint violations. It also supports more plugins and is capable of detecting more errors. However, it does not support custom lint rules like Flake8 does.


> Finished chain.


{'input': 'Why use ruff over flake8?',
 'output': ' Ruff implements more rules than Flake8, has better compatibility with Black, and can automatically fix its own lint violations. It also supports more plugins and is capable of detecting more errors. However, it does not support custom lint rules like Flake8 does.'}

In [27]:
agent.invoke("Summarize the main theme of Sonnet 18.")



> Entering new AgentExecutor chain...
 The Sonnets QA System would be the best tool to use for this question since it is specifically designed for answering questions about poetry.
Action: Sonnets QA System
Action Input: "What is the main theme of Sonnet 18?"
Observation:  The main theme of Sonnet 18 is the speaker's comparison of the subject to a summer's day and the idea that the subject's beauty will never fade.
Thought: This answer seems accurate based on my knowledge of Sonnet 18.
Final Answer: The main theme of Sonnet 18 is the speaker's comparison of the subject to a summer's day and the idea that the subject's beauty will never fade.

> Finished chain.


{'input': 'Summarize the main theme of Sonnet 18.',
 'output': "The main theme of Sonnet 18 is the speaker's comparison of the subject to a summer's day and the idea that the subject's beauty will never fade."}

## Multi-Hop vector store reasoning

Because vector stores are easily usable as tools in agents, it is easy to use answer multi-hop questions that depend on vector stores using the existing agent framework.

In [28]:
tools = [
    Tool(
        name="State of Union QA System",
        func=state_of_union.run,
        description="useful for when you need to answer questions about the most recent state of the union address. Input should be a fully formed question, not referencing any obscure pronouns from the conversation before.",
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question, not referencing any obscure pronouns from the conversation before.",
    ),
    Tool(
        name="Sonnets QA System",
        func=sonnets.run,
        description="useful for answering questions about the sonnets text file (poetry). Ask about lines, themes, imagery, or which sonnet contains a quote. Input should be a fully formed question, not referencing any obscure pronouns from the conversation before.",
    ),
]

In [ ]:
# Construct the agent. We will use the default agent type here.
# See documentation for a full list of options.
agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

In [29]:
agent.invoke(
    "What tool does ruff use to run over Jupyter Notebooks? Did the president mention that tool in the state of the union?"
)



> Entering new AgentExecutor chain...
 I should use the Ruff QA System to answer the first question and the State of Union QA System to answer the second question.
Action: Ruff QA System
Action Input: What tool does ruff use to run over Jupyter Notebooks?
Observation:  Ruff uses nbQA, a tool for running linters and code formatters over Jupyter Notebooks.


> Finished chain.


{'input': 'What tool does ruff use to run over Jupyter Notebooks? Did the president mention that tool in the state of the union?',
 'output': ' Ruff uses nbQA, a tool for running linters and code formatters over Jupyter Notebooks.'}

In [31]:
agent.invoke("""
First, find what tool Ruff uses to run over Jupyter Notebooks.
Then check whether that specific tool is mentioned in the State of the Union.
Answer both parts. and then also the main theme of Sonnet 18. answer all three parts
""")




> Entering new AgentExecutor chain...
 I should first find out what tool Ruff uses to run over Jupyter Notebooks.
Action: Ruff QA System
Action Input: What tool does Ruff use to run over Jupyter Notebooks?
Observation:  Ruff uses nbQA, a tool for running linters and code formatters over Jupyter Notebooks.


> Finished chain.


{'input': '\nFirst, find what tool Ruff uses to run over Jupyter Notebooks.\nThen check whether that specific tool is mentioned in the State of the Union.\nAnswer both parts. and then also the main theme of Sonnet 18. answer all three parts\n',
 'output': ' Ruff uses nbQA, a tool for running linters and code formatters over Jupyter Notebooks.'}

# 

I tried to add the dataset sonnet.text instad of replacing the other dataset. 
tried multi hop vector store reasoning. it outputs only the answer to the first question and then stops. it does not continue to the next question. 
Why Multi-Hop Is Hard


Entity extraction
Memory of intermediate result
Reformulation of second query
Tool chaining
Basic ReAct agents are weak at:
Variable binding across steps
Dependent sub-question generation

#